# O-RAN Version 4 — Dual-Uncertainty QoS Safety-Shield Research Prototype

This notebook bootstraps the complete source package from the committed archive in `source_packages/` if it has not already been extracted. V4 is a simulation research prototype; it does not implement live O-RAN A1/E2 or a production xApp/rApp.

The implementation uses a simulator-consistent conditional Gaussian channel posterior and block-maximum calibration over a predeclared candidate-action family. These choices do not establish a general or simultaneous QoS safety guarantee.

from pathlib import Path
import os, sys, subprocess, tarfile

REPO_ROOT = Path.cwd().resolve()
V4 = REPO_ROOT / "experiments" / "AI-Enabled-O-RAN-ML-DRL-QoS-Optimization-Version4"
if not (V4 / "src" / "evaluate.py").exists():
    archive = REPO_ROOT / "source_packages" / "O-RAN-Version4-Dual-Uncertainty-QoS-Safe-DRL.tar.gz"
    if not archive.exists():
        raise FileNotFoundError(f"Missing V4 source archive: {archive}")
    with tarfile.open(archive, "r:gz") as bundle:
        bundle.extractall(REPO_ROOT / "experiments")
if not (V4 / "src" / "evaluate.py").exists():
    raise FileNotFoundError(f"V4 source package not found after extraction: {V4}")
os.chdir(V4)
sys.path.insert(0, str(V4))
print(f"Using V4 package: {V4}")


## Install dependencies and run software checks

These checks verify package integrity; they do not demonstrate research performance.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "compileall", "-q", "config.py", "src", "tests", "run_all.py"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


In [ ]:
from pathlib import Path
import subprocess, sys
V4 = Path.cwd()
if not (V4/'src'/'safety.py').exists():
    raise FileNotFoundError('Open this notebook with the extracted O-RAN-Version4 folder as the working directory.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Train demand model and conformal upper bounds

In [ ]:
subprocess.run([sys.executable, '-m', 'src.train_demand', '--epochs', '20'], check=True)


## Calibrate receiver-specific channel-rate margins

In [ ]:
subprocess.run([sys.executable, '-m', 'src.calibrate', '--samples', '240'], check=True)


## Train receiver-specific PPO controllers

In [ ]:
for rx in ('MRC', 'ZF', 'MMSE'):
    subprocess.run([sys.executable, '-m', 'src.train_ppo', '--receiver', rx], check=True)


## Evaluate raw PPO and uncertainty shields

In [ ]:
subprocess.run([sys.executable, '-m', 'src.evaluate', '--seeds', '5', '--blocks', '50'], check=True)
import pandas as pd
display(pd.read_csv('results/summary_ci95.csv'))


## Research limitations

The smoke run used only 2 PPO episodes × 5 steps, 12 channel-calibration samples and 2 test seeds × 2 blocks. It passed 4 unit tests but is not research-grade validation. The demand MLP did not beat persistence in the reported 20-epoch synthetic split. Conformal coverage is marginal and assumes exchangeability; it is not a deterministic or simultaneous all-user guarantee. SLSQP is local and may fail. No live O-RAN, A1/E2 or measured hardware power implementation is included.